In [ ]:
---
title: lab_test
description: Lab Model API
show-code: false
---
import mercury as mr
import base64

# 1. نستقبل الصورة كنص بدل ما نستخدم mr.File اللي بيعمل مشاكل
file_b64 = mr.Text(label="file_b64", value="")
file_name = mr.Text(label="file_name", value="uploaded_file.png")

if file_b64.value:
    # 2. بنشيل الزيادات من النص ونحوله لبيانات حقيقية
    b64_string = file_b64.value.split(',')[1] if ',' in file_b64.value else file_b64.value
    
    # 3. بنحفظ الملف على الجهاز باسمه الأصلي
    save_path = file_name.value
    with open(save_path, "wb") as f:
        f.write(base64.b64decode(b64_string))
        
    # ⚠️ الآن: الملف موجود على الجهاز باسم save_path
    # تقدر تباصي الـ save_path للموديل بتاعك عشان يقراه ويطلع النتيجة!
    # مثال: results = model.predict(save_path)

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import joblib

In [8]:
df = pd.read_csv("synthetic_realistic_medical_data_full_labels.csv")
df

,Patient_ID,Gender,WBC,Hemoglobin,Hematocrit,Platelets,Glucose,Calcium,Sodium,Potassium,...,Leukocyte,Diabetes,Hyperlipidemia,Liver_Issue,Hypertension,Kidney_Issue,Anemia,Heart_Disease_Risk,Thyroid_Issue,Infection_Risk
0,P00001,Male,9771.0,14.854144,41.508146,299568.0,82.584548,9.074004,141.702045,4.767347,...,Negative,0,0,0,0,0,0,0,1,0
1,P00002,Female,7246.0,13.456223,38.377918,187338.0,84.810419,9.402895,135.327704,3.970000,...,Negative,0,0,1,0,0,0,0,0,0
2,P00003,Male,6022.0,15.448872,48.314879,284029.0,62.450552,8.607242,139.194080,4.560336,...,Negative,0,0,0,0,0,0,0,0,0
3,P00004,Male,7434.0,17.736304,47.501868,256889.0,85.204562,9.717178,135.503382,3.945897,...,Negative,0,0,0,0,0,0,0,0,0
4,P00005,Male,6424.0,15.484113,NaN,90671.0,NaN,8.837203,140.984560,4.301106,...,Positive,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,P29996,Female,7290.0,12.895090,38.687468,346481.0,209.553228,9.612796,142.385756,4.633048,...,Negative,1,0,0,0,0,0,1,0,0
29996,P29997,Female,6725.0,14.102972,41.915303,367637.0,94.886114,8.955171,143.551566,4.294085,...,Negative,0,0,0,0,0,0,0,0,0
29997,P29998,Female,7600.0,13.286545,39.571012,277035.0,64.143631,9.523517,142.714523,4.016698,...,Negative,0,0,0,0,0,0,0,0,0
29998,P29999,Female,5857.0,13.417356,38.878411,218989.0,90.129036,9.836955,146.660037,4.211023,...,Negative,0,0,0,1,0,0,0,0,0


In [9]:
numeric_cols = ['WBC', 'Hemoglobin', 'Hematocrit', 'Platelets', 'Glucose', 'Calcium',
                'Sodium', 'Potassium', 'Creatinine', 'BUN', 'ALT_AST', 'Albumin',
                'Total_Cholesterol', 'LDL', 'HDL', 'Triglycerides', 'HbA1c', 'TSH']

In [10]:
categorical_cols = ['Gender', 'Protein', 'Urine_Glucose', 'Nitrites', 'Leukocyte']

In [11]:
target_cols = ['Diabetes', 'Hyperlipidemia', 'Liver_Issue', 'Hypertension', 
               'Kidney_Issue', 'Anemia', 'Heart_Disease_Risk', 'Thyroid_Issue', 'Infection_Risk']

In [12]:
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [13]:
for col in target_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [14]:
X = df[numeric_cols + categorical_cols]
y = df[target_cols]

In [15]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [16]:
model = MultiOutputClassifier(HistGradientBoostingClassifier(random_state=42))

pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [18]:
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['WBC', 'Hemoglobin',
                                                   'Hematocrit', 'Platelets',
                                                   'Glucose', 'Calcium',
                                                   'Sodium', 'Potassium',
                                                   'Creatinine', 'BUN',
                                                   'ALT_AST', 'Albumin',
                                                   'Total_Cholesterol', 'LDL',
                                                   'HDL', 'Triglycerides',
                                                   'HbA1c', 'TSH']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'Protein',
                                                   'Urine_Glucose', 'Nitrites',
                                                   'Leukocyte'])])),
                ('classifier',
                 MultiOutputClassifier(estimator=HistGradientBoostingClassifier(random_state=42)))])

In [19]:
y_pred = pipeline.predict(X_test)
print('--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=target_cols))

print('--- Confusion Matrices ---')
mcm = multilabel_confusion_matrix(y_test, y_pred)
for idx, col in enumerate(target_cols):
    print(f'Confusion Matrix for {col}:')
    print(mcm[idx])

--- Classification Report ---
                    precision    recall  f1-score   support

          Diabetes       0.98      1.00      0.99       807
    Hyperlipidemia       1.00      0.99      0.99       438
       Liver_Issue       0.98      1.00      0.99        80
      Hypertension       0.99      0.98      0.99       399
      Kidney_Issue       0.94      0.93      0.93        97
            Anemia       0.98      0.99      0.99       309
Heart_Disease_Risk       1.00      1.00      1.00      1458
     Thyroid_Issue       0.98      1.00      0.99       456
    Infection_Risk       0.99      0.97      0.98       232

         micro avg       0.99      0.99      0.99      4276
         macro avg       0.98      0.98      0.98      4276
      weighted avg       0.99      0.99      0.99      4276
       samples avg       0.51      0.51      0.51      4276

--- Confusion Matrices ---
Confusion Matrix for Diabetes:
[[5180   13]
 [   4  803]]
Confusion Matrix for Hyperlipidemia:
[[556

c:\Users\Youssef\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Youssef\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Youssef\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [20]:
y_test_pred = pipeline.predict(X_test)
print('--- Classification Report on Test Set ---')
print(classification_report(y_test, y_test_pred, target_names=target_cols))

print('--- Confusion Matrices on Test Set ---')
mcm_test = multilabel_confusion_matrix(y_test, y_test_pred)
for idx, col in enumerate(target_cols):
    print(f'Confusion Matrix for {col} (Test):')
    print(mcm_test[idx])

--- Classification Report on Test Set ---
                    precision    recall  f1-score   support

          Diabetes       0.98      1.00      0.99       807
    Hyperlipidemia       1.00      0.99      0.99       438
       Liver_Issue       0.98      1.00      0.99        80
      Hypertension       0.99      0.98      0.99       399
      Kidney_Issue       0.94      0.93      0.93        97
            Anemia       0.98      0.99      0.99       309
Heart_Disease_Risk       1.00      1.00      1.00      1458
     Thyroid_Issue       0.98      1.00      0.99       456
    Infection_Risk       0.99      0.97      0.98       232

         micro avg       0.99      0.99      0.99      4276
         macro avg       0.98      0.98      0.98      4276
      weighted avg       0.99      0.99      0.99      4276
       samples avg       0.51      0.51      0.51      4276

--- Confusion Matrices on Test Set ---
Confusion Matrix for Diabetes (Test):
[[5180   13]
 [   4  803]]
Confusion M

c:\Users\Youssef\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Youssef\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Youssef\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [21]:
joblib.dump(pipeline, 'lab_tests_model.pkl')
print('Model saved as lab_tests_model.pkl')

Model saved as lab_tests_model.pkl
